# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Storage Solutions (MongoDB)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from SparkUtils import SparkUtils

import pyspark.sql.functions as F

In [2]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Example: MongoDB"
SPARK_PACKAGES = "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0"

spark = SparkUtils(MASTER_URL, APP_NAME, spark_packages=SPARK_PACKAGES)._spark

spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ce2efa8c-174a-4ff0-b060-d63ae17a8c38;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.5.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.13/10.5.0/mongo-spark-connector_2.13-10.5.0.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.13;10.5.0!mongo-spark-connector_2.13.jar (114ms)
downloading 

# Create DataFrames
## Videogames dataframe

In [3]:
schema = SparkUtils.generate_schema([
    ("title",     "string"),
    ("platform",  "string"),
    ("rating",    "double"),
    ("reviews",   "int"),
    ("tags",      "array_string"),
    ("publisher", "struct", [
        ("name",    "string"),
        ("country", "string"),
    ])
])

data = [
    ("The Legend of Zelda", "Switch", 4.9, 3400, ["adventure","RPG","open-world"], ("Nintendo","Japan")),
    ("God of War", "PS5", 4.8, 2800, ["action","adventure","story"], ("Sony","USA")),
    ("Halo Infinite", "Xbox", 4.3, 1500, ["FPS","multiplayer","sci-fi"], ("Microsoft","USA")),
    ("Stardew Valley", "PC", 4.7, 5200, ["simulation","indie","farming"], ("ConcernedApe","USA")),
]

videogames_df = spark.createDataFrame(data, schema)

videogames_df.show(truncate=False)

videogames_df.printSchema()

+-------------------+--------+------+-------+----------------------------+-------------------+
|title              |platform|rating|reviews|tags                        |publisher          |
+-------------------+--------+------+-------+----------------------------+-------------------+
|The Legend of Zelda|Switch  |4.9   |3400   |[adventure, RPG, open-world]|{Nintendo, Japan}  |
|God of War         |PS5     |4.8   |2800   |[action, adventure, story]  |{Sony, USA}        |
|Halo Infinite      |Xbox    |4.3   |1500   |[FPS, multiplayer, sci-fi]  |{Microsoft, USA}   |
|Stardew Valley     |PC      |4.7   |5200   |[simulation, indie, farming]|{ConcernedApe, USA}|
+-------------------+--------+------+-------+----------------------------+-------------------+

root
 |-- title: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = tr

## Users and Ratings DataFrames

In [4]:
users_data = [
    ("u001", "Alice",  "alice@mail.com",  "2024-01-15"),
    ("u002", "Bob",    "bob@mail.com",    "2024-02-20"),
    ("u003", "Carol",  "carol@mail.com",  "2023-11-05"),
    ("u004", "David",  "david@mail.com",  "2024-03-10"),
]

users_schema = SparkUtils.generate_schema([
    ("user_id", "string"),
    ("name", "string"),
    ("e-mail", "string"),
    ("signup_date_str", "string")
])

users_df = spark.createDataFrame(users_data, users_schema)

users_df = users_df.withColumn("signup_date", F.to_date("signup_date_str", "yyyy-MM-dd")).drop("signup_date_str")

users_df.show()

+-------+-----+--------------+-----------+
|user_id| name|        e-mail|signup_date|
+-------+-----+--------------+-----------+
|   u001|Alice|alice@mail.com| 2024-01-15|
|   u002|  Bob|  bob@mail.com| 2024-02-20|
|   u003|Carol|carol@mail.com| 2023-11-05|
|   u004|David|david@mail.com| 2024-03-10|
+-------+-----+--------------+-----------+



In [5]:
ratings_data = [
    ("u001", "The Legend of Zelda", 5.0, "2024-06-01"),
    ("u001", "Stardew Valley",     4.5, "2024-06-15"),
    ("u002", "God of War",         4.8, "2024-07-01"),
    ("u003", "Halo Infinite",      3.9, "2024-05-20"),
    ("u003", "The Legend of Zelda", 4.7, "2024-05-25"),
    ("u004", "Stardew Valley",     5.0, "2024-08-01"),
]

ratings_schema = SparkUtils.generate_schema([
    ("user_id", "string"),
    ("game_title", "string"),
    ("score", "float"),
    ("date_str", "string")
])

ratings_df = spark.createDataFrame(ratings_data, ratings_schema)

ratings_df = ratings_df.withColumn("rate_date", F.to_date("date_str", "yyyy-MM-dd")).drop("date_str")

ratings_df.show()

+-------+-------------------+-----+----------+
|user_id|         game_title|score| rate_date|
+-------+-------------------+-----+----------+
|   u001|The Legend of Zelda|  5.0|2024-06-01|
|   u001|     Stardew Valley|  4.5|2024-06-15|
|   u002|         God of War|  4.8|2024-07-01|
|   u003|      Halo Infinite|  3.9|2024-05-20|
|   u003|The Legend of Zelda|  4.7|2024-05-25|
|   u004|     Stardew Valley|  5.0|2024-08-01|
+-------+-------------------+-----+----------+



# Transformations and Aggregations

In [6]:
game_stats = ratings_df \
    .groupBy("game_title") \
    .agg(
        F.round(F.avg("score"), 2).alias("avg_score"),
        F.count("*").alias("num_ratings"),
        F.max("rate_date").alias("last_rated"),
    ) \
    .orderBy(F.desc("avg_score"))

game_stats.show()

# enriches users with their rating history as an array
user_ratings = ratings_df \
    .groupBy("user_id") \
    .agg(
        F.collect_list(
            F.struct("game_title", "score", "rate_date")
        ).alias("ratings")
    )

enriched_users = users_df.join(user_ratings, on="user_id", how="left")

enriched_users.printSchema()

enriched_users.show(truncate=False)

+-------------------+---------+-----------+----------+
|         game_title|avg_score|num_ratings|last_rated|
+-------------------+---------+-----------+----------+
|The Legend of Zelda|     4.85|          2|2024-06-01|
|         God of War|      4.8|          1|2024-07-01|
|     Stardew Valley|     4.75|          2|2024-08-01|
|      Halo Infinite|      3.9|          1|2024-05-20|
+-------------------+---------+-----------+----------+

root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- e-mail: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- ratings: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |    |    |-- game_title: string (nullable = true)
 |    |    |-- score: float (nullable = true)
 |    |    |-- rate_date: date (nullable = true)



+-------+-----+--------------+-----------+---------------------------------------------------------------------------+
|user_id|name |e-mail        |signup_date|ratings                                                                    |
+-------+-----+--------------+-----------+---------------------------------------------------------------------------+
|u001   |Alice|alice@mail.com|2024-01-15 |[{The Legend of Zelda, 5.0, 2024-06-01}, {Stardew Valley, 4.5, 2024-06-15}]|
|u002   |Bob  |bob@mail.com  |2024-02-20 |[{God of War, 4.8, 2024-07-01}]                                            |
|u004   |David|david@mail.com|2024-03-10 |[{Stardew Valley, 5.0, 2024-08-01}]                                        |
|u003   |Carol|carol@mail.com|2023-11-05 |[{Halo Infinite, 3.9, 2024-05-20}, {The Legend of Zelda, 4.7, 2024-05-25}] |
+-------+-----+--------------+-----------+---------------------------------------------------------------------------+



# Write to MongoDB

In [7]:
mongo_uri = "mongodb://mongodb-pdm:27017"

enriched_users.write \
    .format("mongodb") \
    .option("database", "videogames") \
    .option("collection", "ratings") \
    .option("connection.uri", mongo_uri) \
    .mode("overwrite") \
    .save()

### Inspecting MongoDB Collections in Docker
```bash
# 1. Open a shell inside the container
docker exec -it <container_name> mongosh
```
```javascript
// 2. List all databases
show dbs

// 3. Switch to your database
use <database_name>

// 4. List all collections
show collections

// 5. Count documents in a collection
db.<collection_name>.countDocuments()

// 6. Preview documents
db.<collection_name>.find().limit(5).pretty()
```

![image.png](../img/lab-07-documents-preview.png)

In [ ]:
# spark.stop()